In [1]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [2]:
pred_data = pd.read_csv('pred_data\CIS_662 _INITIAL_Predictions.csv')

In [3]:
pred_data.head()

,DATE,DAY,FLIGHT NUMBER,ORIGIN,DEPARTURE TIME,ARRIVAL TIME,ARRIVAL STATUS,ARRIVAL STATUS_Prev_flight_early,ARRIVAL STATUS_Prev_flight_ontime,ARRIVAL STATUS_Prev_flight_late
0,4/10/24,WEDNESDAY,UA 1400,ORD,6:52 PM,9:47 PM,NaN,NaN,NaN,NaN
1,4/10/24,WEDNESDAY,AA 3402,ORD,7:59 PM,10:52 PM,NaN,NaN,NaN,NaN
2,4/10/24,WEDNESDAY,B6 116,JFK,1:33 PM,2:50 PM,NaN,NaN,NaN,NaN
3,4/10/24,WEDNESDAY,DL 5182,JFK,2:55 PM,4:21 PM,NaN,NaN,NaN,NaN
4,4/10/24,WEDNESDAY,WN 5285,MCO,11:05 AM,1:45 PM,NaN,NaN,NaN,NaN


In [4]:
pred_data.dtypes

DATE                                  object
DAY                                   object
FLIGHT NUMBER                         object
ORIGIN                                object
DEPARTURE TIME                        object
ARRIVAL TIME                          object
ARRIVAL STATUS                       float64
ARRIVAL STATUS_Prev_flight_early     float64
ARRIVAL STATUS_Prev_flight_ontime    float64
ARRIVAL STATUS_Prev_flight_late      float64
dtype: object

In [5]:
pred_data['DATE'] = pd.to_datetime( pred_data['DATE'],format ="%m/%d/%y")
pred_data['DAY'] = pred_data['DATE'].dt.day_of_week.astype('object')
pred_data['Carrier_Code'] = pred_data['FLIGHT NUMBER'].str[0:2].astype('object')
pred_data['DEPARTURE TIME'] = pd.to_datetime(pred_data['DEPARTURE TIME'], format = '%I:%M %p')
pred_data['dep_hour']= pred_data['DEPARTURE TIME'].dt.hour.astype('object')
pred_data['dep_min'] = pred_data['DEPARTURE TIME'].dt.minute.astype('object')
pred_data['ARRIVAL TIME'] = pd.to_datetime(pred_data['ARRIVAL TIME'], format = '%I:%M %p')
pred_data['arr_hour']= pred_data['ARRIVAL TIME'].dt.hour.astype('object')
pred_data['arr_min'] = pred_data['ARRIVAL TIME'].dt.minute.astype('object')
pred_data.rename(columns = {'ORIGIN' : 'Origin_Airport'},inplace =True)
pred_data.head()

,DATE,DAY,FLIGHT NUMBER,Origin_Airport,DEPARTURE TIME,ARRIVAL TIME,ARRIVAL STATUS,ARRIVAL STATUS_Prev_flight_early,ARRIVAL STATUS_Prev_flight_ontime,ARRIVAL STATUS_Prev_flight_late,Carrier_Code,dep_hour,dep_min,arr_hour,arr_min
0,2024-04-10,2,UA 1400,ORD,1900-01-01 18:52:00,1900-01-01 21:47:00,NaN,NaN,NaN,NaN,UA,18,52,21,47
1,2024-04-10,2,AA 3402,ORD,1900-01-01 19:59:00,1900-01-01 22:52:00,NaN,NaN,NaN,NaN,AA,19,59,22,52
2,2024-04-10,2,B6 116,JFK,1900-01-01 13:33:00,1900-01-01 14:50:00,NaN,NaN,NaN,NaN,B6,13,33,14,50
3,2024-04-10,2,DL 5182,JFK,1900-01-01 14:55:00,1900-01-01 16:21:00,NaN,NaN,NaN,NaN,DL,14,55,16,21
4,2024-04-10,2,WN 5285,MCO,1900-01-01 11:05:00,1900-01-01 13:45:00,NaN,NaN,NaN,NaN,WN,11,5,13,45


In [6]:
pred_data.dtypes

DATE                                 datetime64[ns]
DAY                                          object
FLIGHT NUMBER                                object
Origin_Airport                               object
DEPARTURE TIME                       datetime64[ns]
ARRIVAL TIME                         datetime64[ns]
ARRIVAL STATUS                              float64
ARRIVAL STATUS_Prev_flight_early            float64
ARRIVAL STATUS_Prev_flight_ontime           float64
ARRIVAL STATUS_Prev_flight_late             float64
Carrier_Code                                 object
dep_hour                                     object
dep_min                                      object
arr_hour                                     object
arr_min                                      object
dtype: object

In [7]:
conditions = [
    (pred_data['FLIGHT NUMBER'] == 'UA 1400'),
    (pred_data['FLIGHT NUMBER'] == 'AA 3402'),
    (pred_data['FLIGHT NUMBER'] == 'B6 116'),
    (pred_data['FLIGHT NUMBER'] == 'DL 5182'),
    (pred_data['FLIGHT NUMBER'] == 'WN 5285'),
    (pred_data['FLIGHT NUMBER'] == 'B6 656')
]
choices = ['early','latter','early','latter','early','latter']
pred_data['dep_order'] = np.select(conditions, choices)

In [8]:
pred_data.head(30)

,DATE,DAY,FLIGHT NUMBER,Origin_Airport,DEPARTURE TIME,ARRIVAL TIME,ARRIVAL STATUS,ARRIVAL STATUS_Prev_flight_early,ARRIVAL STATUS_Prev_flight_ontime,ARRIVAL STATUS_Prev_flight_late,Carrier_Code,dep_hour,dep_min,arr_hour,arr_min,dep_order
0,2024-04-10,2,UA 1400,ORD,1900-01-01 18:52:00,1900-01-01 21:47:00,NaN,NaN,NaN,NaN,UA,18,52,21,47,early
1,2024-04-10,2,AA 3402,ORD,1900-01-01 19:59:00,1900-01-01 22:52:00,NaN,NaN,NaN,NaN,AA,19,59,22,52,latter
2,2024-04-10,2,B6 116,JFK,1900-01-01 13:33:00,1900-01-01 14:50:00,NaN,NaN,NaN,NaN,B6,13,33,14,50,early
3,2024-04-10,2,DL 5182,JFK,1900-01-01 14:55:00,1900-01-01 16:21:00,NaN,NaN,NaN,NaN,DL,14,55,16,21,latter
4,2024-04-10,2,WN 5285,MCO,1900-01-01 11:05:00,1900-01-01 13:45:00,NaN,NaN,NaN,NaN,WN,11,5,13,45,early
5,2024-04-10,2,B6 656,MCO,1900-01-01 13:35:00,1900-01-01 16:25:00,NaN,NaN,NaN,NaN,B6,13,35,16,25,latter
6,2024-04-11,3,UA 1400,ORD,1900-01-01 18:52:00,1900-01-01 21:47:00,NaN,NaN,NaN,NaN,UA,18,52,21,47,early
7,2024-04-11,3,AA 3402,ORD,1900-01-01 19:59:00,1900-01-01 22:52:00,NaN,NaN,NaN,NaN,AA,19,59,22,52,latter
8,2024-04-11,3,B6 116,JFK,1900-01-01 13:33:00,1900-01-01 14:50:00,NaN,NaN,NaN,NaN,B6,13,33,14,50,early
9,2024-04-11,3,DL 5182,JFK,1900-01-01 14:55:00,1900-01-01 16:21:00,NaN,NaN,NaN,NaN,DL,14,55,16,21,latter


In [9]:
pred_data['dep_minutes'] = 0
pred_data['dep_minutes'] = pred_data['dep_minutes'].astype('object')
pred_data['dep_hours'] = pred_data['dep_hour'].astype('object')
pred_data['arr_minutes'] = 0
pred_data['arr_minutes'] = pred_data['arr_minutes'].astype('object')
pred_data['arr_hours'] = pred_data['arr_hour'].astype('object')
pred_data.rename(columns ={'DATE':'Date'},inplace=True)
pred_data.head()


,Date,DAY,FLIGHT NUMBER,Origin_Airport,DEPARTURE TIME,ARRIVAL TIME,ARRIVAL STATUS,ARRIVAL STATUS_Prev_flight_early,ARRIVAL STATUS_Prev_flight_ontime,ARRIVAL STATUS_Prev_flight_late,Carrier_Code,dep_hour,dep_min,arr_hour,arr_min,dep_order,dep_minutes,dep_hours,arr_minutes,arr_hours
0,2024-04-10,2,UA 1400,ORD,1900-01-01 18:52:00,1900-01-01 21:47:00,NaN,NaN,NaN,NaN,UA,18,52,21,47,early,0,18,0,21
1,2024-04-10,2,AA 3402,ORD,1900-01-01 19:59:00,1900-01-01 22:52:00,NaN,NaN,NaN,NaN,AA,19,59,22,52,latter,0,19,0,22
2,2024-04-10,2,B6 116,JFK,1900-01-01 13:33:00,1900-01-01 14:50:00,NaN,NaN,NaN,NaN,B6,13,33,14,50,early,0,13,0,14
3,2024-04-10,2,DL 5182,JFK,1900-01-01 14:55:00,1900-01-01 16:21:00,NaN,NaN,NaN,NaN,DL,14,55,16,21,latter,0,14,0,16
4,2024-04-10,2,WN 5285,MCO,1900-01-01 11:05:00,1900-01-01 13:45:00,NaN,NaN,NaN,NaN,WN,11,5,13,45,early,0,11,0,13


In [10]:
# pred_data['Date'] = pred_data['Date'].astype('object')
# datetime_obj = pd.to_datetime(pred_data['Date'])
# pred_data['Date']=datetime_obj.dt.date

pred_data['Date'] = pred_data['Date'].dt.strftime('%m/%d/%Y')
pred_data.dtypes
len(pred_data)
pred_data.head(30)

Date                                         object
DAY                                          object
FLIGHT NUMBER                                object
Origin_Airport                               object
DEPARTURE TIME                       datetime64[ns]
ARRIVAL TIME                         datetime64[ns]
ARRIVAL STATUS                              float64
ARRIVAL STATUS_Prev_flight_early            float64
ARRIVAL STATUS_Prev_flight_ontime           float64
ARRIVAL STATUS_Prev_flight_late             float64
Carrier_Code                                 object
dep_hour                                     object
dep_min                                      object
arr_hour                                     object
arr_min                                      object
dep_order                                    object
dep_minutes                                  object
dep_hours                                    object
arr_minutes                                  object
arr_hours   

23

,Date,DAY,FLIGHT NUMBER,Origin_Airport,DEPARTURE TIME,ARRIVAL TIME,ARRIVAL STATUS,ARRIVAL STATUS_Prev_flight_early,ARRIVAL STATUS_Prev_flight_ontime,ARRIVAL STATUS_Prev_flight_late,Carrier_Code,dep_hour,dep_min,arr_hour,arr_min,dep_order,dep_minutes,dep_hours,arr_minutes,arr_hours
0,04/10/2024,2,UA 1400,ORD,1900-01-01 18:52:00,1900-01-01 21:47:00,NaN,NaN,NaN,NaN,UA,18,52,21,47,early,0,18,0,21
1,04/10/2024,2,AA 3402,ORD,1900-01-01 19:59:00,1900-01-01 22:52:00,NaN,NaN,NaN,NaN,AA,19,59,22,52,latter,0,19,0,22
2,04/10/2024,2,B6 116,JFK,1900-01-01 13:33:00,1900-01-01 14:50:00,NaN,NaN,NaN,NaN,B6,13,33,14,50,early,0,13,0,14
3,04/10/2024,2,DL 5182,JFK,1900-01-01 14:55:00,1900-01-01 16:21:00,NaN,NaN,NaN,NaN,DL,14,55,16,21,latter,0,14,0,16
4,04/10/2024,2,WN 5285,MCO,1900-01-01 11:05:00,1900-01-01 13:45:00,NaN,NaN,NaN,NaN,WN,11,5,13,45,early,0,11,0,13
5,04/10/2024,2,B6 656,MCO,1900-01-01 13:35:00,1900-01-01 16:25:00,NaN,NaN,NaN,NaN,B6,13,35,16,25,latter,0,13,0,16
6,04/11/2024,3,UA 1400,ORD,1900-01-01 18:52:00,1900-01-01 21:47:00,NaN,NaN,NaN,NaN,UA,18,52,21,47,early,0,18,0,21
7,04/11/2024,3,AA 3402,ORD,1900-01-01 19:59:00,1900-01-01 22:52:00,NaN,NaN,NaN,NaN,AA,19,59,22,52,latter,0,19,0,22
8,04/11/2024,3,B6 116,JFK,1900-01-01 13:33:00,1900-01-01 14:50:00,NaN,NaN,NaN,NaN,B6,13,33,14,50,early,0,13,0,14
9,04/11/2024,3,DL 5182,JFK,1900-01-01 14:55:00,1900-01-01 16:21:00,NaN,NaN,NaN,NaN,DL,14,55,16,21,latter,0,14,0,16


In [11]:
# Read and process weather data files for each airport
jfk_weather_data = pd.read_csv('weather_data/JFK_weather_data_forecast_processed.csv')
syr_weather_data = pd.read_csv('weather_data/SYR_weather_data_forecast_processed.csv')
ord_weather_data = pd.read_csv('weather_data/ORD_weather_data_forecast_processed.csv')
mco_weather_data = pd.read_csv('weather_data/MCO_weather_data_forecast_processed.csv')

# Combine weather data for all airports
weather_dfs = [jfk_weather_data, ord_weather_data, mco_weather_data]
weather_data = pd.concat(weather_dfs, axis=0)
weather_data['dep_minutes'] = weather_data['dep_minutes'].astype('object')
weather_data['dep_hours'] = weather_data['dep_hours'].astype('object')
syr_weather_data['arr_minutes'] = syr_weather_data['arr_minutes'].astype('object')
syr_weather_data['arr_hours'] = syr_weather_data['arr_hours'].astype('object')
weather_data.head(50)
len(weather_data)
len(syr_weather_data)

# Define merging logic based on airport code
pred_data = pd.merge(pred_data, syr_weather_data, how='left', on=['Date', 'arr_hours', 'arr_minutes'])
len(pred_data)
pred_data.to_csv("first.csv")

pred_data = pd.merge(pred_data, weather_data, how='left', on=['Origin_Airport', 'Date', 'dep_hours', 'dep_minutes'])
pred_data.to_csv("sec.csv")
len(pred_data)


#weather_data.head()


,dep_clouds,dep_clouds_hi,dep_clouds_low,dep_clouds_mid,dep_datetime,dep_dewpt,dep_ozone,dep_pop,dep_precip,dep_pres,...,dep_weather.code,dep_wind_cdir,dep_wind_cdir_full,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd,Date,dep_hours,dep_minutes,Origin_Airport
0,80,6,100,100,2024-04-09:01,0.80,355.00,0,0.00,"1,019.50",...,804,SSW,south-southwest,200,4.40,3.20,04/08/2024,21,0,JFK
1,81,0,100,100,2024-04-09:02,5.00,354.00,0,0.00,"1,018.50",...,804,SSW,south-southwest,210,3.60,2.40,04/08/2024,22,0,JFK
2,71,99,0,2,2024-04-09:03,4.80,355.00,0,0.00,"1,018.00",...,804,SW,southwest,220,3.60,2.40,04/08/2024,23,0,JFK
3,76,2,3,54,2024-04-09:04,5.30,357.80,0,0.00,"1,017.50",...,804,SW,southwest,220,3.60,2.40,04/09/2024,0,0,JFK
4,79,0,20,15,2024-04-09:05,5.20,359.50,0,0.00,"1,017.50",...,804,SW,southwest,230,3.20,2.00,04/09/2024,1,0,JFK
5,73,0,41,4,2024-04-09:06,5.20,362.80,0,0.00,"1,018.00",...,804,SW,southwest,230,3.20,2.00,04/09/2024,2,0,JFK
6,61,0,39,2,2024-04-09:07,5.40,364.80,0,0.00,"1,018.00",...,803,SW,southwest,220,2.80,2.00,04/09/2024,3,0,JFK
7,55,0,43,2,2024-04-09:08,5.80,368.00,0,0.00,"1,017.50",...,803,SSW,south-southwest,210,2.40,1.60,04/09/2024,4,0,JFK
8,29,0,11,0,2024-04-09:09,5.50,365.80,0,0.00,"1,017.00",...,802,SSW,south-southwest,210,2.80,1.60,04/09/2024,5,0,JFK
9,18,1,8,0,2024-04-09:10,5.50,366.00,0,0.00,"1,017.00",...,801,SW,southwest,220,2.00,1.60,04/09/2024,6,0,JFK


504

168

23

23

In [12]:
pred_data.to_csv('pred_mid.csv')

In [13]:
pred_data['arr_day'] = pred_data['DAY']
pred_data['dep_day'] = pred_data['DAY']
pred_data.drop(columns = ['DAY','DEPARTURE TIME','ARRIVAL TIME','ARRIVAL STATUS_Prev_flight_early',
                         'ARRIVAL STATUS_Prev_flight_ontime','ARRIVAL STATUS_Prev_flight_late',
                         'FLIGHT NUMBER','Date','ARRIVAL STATUS'],inplace = True)



In [14]:
pred_data.dtypes

Origin_Airport              object
Carrier_Code                object
dep_hour                    object
dep_min                     object
arr_hour                    object
arr_min                     object
dep_order                   object
dep_minutes                 object
dep_hours                   object
arr_minutes                 object
arr_hours                   object
arr_clouds                   int64
arr_clouds_hi                int64
arr_clouds_low               int64
arr_clouds_mid               int64
arr_datetime                object
arr_dewpt                  float64
arr_ozone                  float64
arr_pop                      int64
arr_precip                 float64
arr_pres                   float64
arr_rh                       int64
arr_snow                     int64
arr_snow_depth               int64
arr_temp                   float64
arr_vis                    float64
arr_weather.description     object
arr_weather.code             int64
arr_wind_cdir       

In [15]:
pred_data['dep_hour'] = pd.Categorical(pred_data['dep_hour'], categories=[i for i in range(24)])
pred_data['dep_day'] = pd.Categorical(pred_data['dep_day'], categories=[i for i in range(7)])
pred_data['dep_min'] = pd.Categorical(pred_data['dep_min'], categories=[i for i in range(60)])
pred_data['arr_hour'] = pd.Categorical(pred_data['arr_hour'], categories=[i for i in range(24)])
pred_data['arr_day'] = pd.Categorical(pred_data['arr_day'], categories=[i for i in range(7)])
pred_data['arr_min'] = pd.Categorical(pred_data['arr_min'], categories=[i for i in range(60)])
pred_data['Carrier_Code'] = pd.Categorical(pred_data['Carrier_Code'], categories=['AA', 'UA', 'DL', 'B6', 'WN'])
pred_data['Origin_Airport'] = pd.Categorical(pred_data['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])
pred_data=pred_data[['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'arr_clouds', 'arr_dewpt',
       'arr_precip', 'arr_pres', 'arr_rh', 'arr_snow', 'arr_temp', 'arr_vis',
       'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd', 'arr_wind_spd',
       'dep_clouds', 'dep_dewpt', 'dep_precip', 'dep_pres', 'dep_rh',
       'dep_snow', 'dep_temp', 'dep_vis', 'dep_weather.code', 'dep_wind_dir',
       'dep_wind_gust_spd', 'dep_wind_spd','dep_order']]
pred_data.head()
pred_data.columns

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,arr_clouds,arr_dewpt,arr_precip,...,dep_pres,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd,dep_order
0,18,2,ORD,21,2,52,47,77,7.00,0.25,...,983.50,43,0,16.40,24.13,804,73,4.43,3.33,early
1,19,2,ORD,22,2,59,52,76,6.60,0.25,...,982.50,45,0,15.50,24.13,804,70,4.20,3.20,latter
2,13,2,JFK,14,2,33,50,84,9.40,0.25,...,"1,015.00",77,0,11.90,0.10,804,96,4.33,3.60,early
3,14,2,JFK,16,2,55,21,80,8.80,0.25,...,"1,014.50",76,0,12.40,0.20,804,90,4.10,3.60,latter
4,11,2,MCO,13,2,5,45,86,9.50,0.25,...,"1,015.00",63,0,25.20,26.50,803,150,9.60,6.40,early


Index(['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'arr_clouds', 'arr_dewpt', 'arr_precip',
       'arr_pres', 'arr_rh', 'arr_snow', 'arr_temp', 'arr_vis',
       'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd', 'arr_wind_spd',
       'dep_clouds', 'dep_dewpt', 'dep_precip', 'dep_pres', 'dep_rh',
       'dep_snow', 'dep_temp', 'dep_vis', 'dep_weather.code', 'dep_wind_dir',
       'dep_wind_gust_spd', 'dep_wind_spd', 'dep_order'],
      dtype='object')

In [16]:
pred_data1 = pred_data[pred_data['dep_order'] == 'early']
pred_data2 = pred_data[pred_data['dep_order'] == 'latter']


In [17]:
pred_data1.drop(columns=['dep_order'],inplace = True)
pred_data2.drop(columns=['dep_order'],inplace = True)

C:\Users\gurud\AppData\Local\Temp\ipykernel_29888\2195326561.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pred_data1.drop(columns=['dep_order'],inplace = True)
C:\Users\gurud\AppData\Local\Temp\ipykernel_29888\2195326561.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pred_data2.drop(columns=['dep_order'],inplace = True)


In [18]:
pred_data1.head(22)

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,arr_clouds,arr_dewpt,arr_precip,...,dep_precip,dep_pres,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,2,ORD,21,2,52,47,77,7.00,0.25,...,0.00,983.50,43,0,16.40,24.13,804,73,4.43,3.33
2,13,2,JFK,14,2,33,50,84,9.40,0.25,...,0.25,"1,015.00",77,0,11.90,0.10,804,96,4.33,3.60
4,11,2,MCO,13,2,5,45,86,9.50,0.25,...,0.00,"1,015.00",63,0,25.20,26.50,803,150,9.60,6.40
6,18,3,ORD,21,3,52,47,89,11.40,0.76,...,0.58,971.50,70,0,10.90,24.10,301,323,16.93,11.33
8,13,3,JFK,14,3,33,50,92,11.10,0.50,...,0.25,"1,011.00",81,0,12.70,24.13,804,140,9.46,6.26
10,11,3,MCO,14,3,35,20,92,11.10,0.50,...,0.25,"1,008.50",66,0,27.80,24.13,803,180,15.20,10.00
12,18,4,ORD,21,4,52,47,76,3.20,0.25,...,0.00,986.50,49,0,12.20,24.13,802,300,14.55,9.90
14,13,4,JFK,14,4,33,50,82,7.00,0.50,...,0.62,992.50,67,0,15.70,24.13,301,213,15.85,10.79
16,11,4,MCO,14,4,35,20,82,7.00,0.50,...,0.00,"1,018.00",60,0,21.30,24.13,801,305,8.75,5.63
18,18,5,ORD,21,5,52,47,75,1.10,0.00,...,0.00,981.70,47,0,16.60,24.13,802,206,8.91,5.81


In [19]:
pred_data2.head(20)

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,arr_clouds,arr_dewpt,arr_precip,...,dep_precip,dep_pres,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
1,19,2,ORD,22,2,59,52,76,6.60,0.25,...,0.00,982.50,45,0,15.50,24.13,804,70,4.20,3.20
3,14,2,JFK,16,2,55,21,80,8.80,0.25,...,0.25,"1,014.50",76,0,12.40,0.20,804,90,4.10,3.60
5,13,2,MCO,16,2,35,25,80,8.80,0.25,...,0.00,"1,014.50",52,0,27.60,31.70,803,150,9.66,6.40
7,19,3,ORD,22,3,59,52,89,11.50,0.76,...,0.50,971.50,70,0,10.40,24.13,804,320,17.20,11.60
9,14,3,JFK,16,3,55,21,87,11.00,0.50,...,0.25,"1,010.00",80,0,13.20,24.13,804,150,10.80,7.20
11,13,3,MCO,16,3,35,25,87,11.00,0.50,...,0.25,"1,008.00",57,0,29.80,24.13,803,193,18.13,11.86
13,19,4,ORD,22,4,59,52,78,3.10,0.25,...,0.00,987.50,47,0,12.30,24.13,801,300,14.40,9.87
15,14,4,JFK,16,4,55,21,79,5.90,0.41,...,0.50,993.50,63,0,16.10,24.13,803,220,15.30,10.47
17,13,4,MCO,16,4,35,25,79,5.90,0.41,...,0.00,"1,017.00",45,0,23.90,24.13,801,301,9.11,5.92
19,19,5,ORD,22,5,59,52,74,1.30,0.00,...,0.00,980.50,48,0,16.60,24.13,802,200,8.90,5.80


In [20]:
pred_data1.to_csv('pred_data1.csv')

In [21]:
len(pred_data1)

11

In [22]:
pred_data1.head(20)

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,arr_clouds,arr_dewpt,arr_precip,...,dep_precip,dep_pres,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,2,ORD,21,2,52,47,77,7.00,0.25,...,0.00,983.50,43,0,16.40,24.13,804,73,4.43,3.33
2,13,2,JFK,14,2,33,50,84,9.40,0.25,...,0.25,"1,015.00",77,0,11.90,0.10,804,96,4.33,3.60
4,11,2,MCO,13,2,5,45,86,9.50,0.25,...,0.00,"1,015.00",63,0,25.20,26.50,803,150,9.60,6.40
6,18,3,ORD,21,3,52,47,89,11.40,0.76,...,0.58,971.50,70,0,10.90,24.10,301,323,16.93,11.33
8,13,3,JFK,14,3,33,50,92,11.10,0.50,...,0.25,"1,011.00",81,0,12.70,24.13,804,140,9.46,6.26
10,11,3,MCO,14,3,35,20,92,11.10,0.50,...,0.25,"1,008.50",66,0,27.80,24.13,803,180,15.20,10.00
12,18,4,ORD,21,4,52,47,76,3.20,0.25,...,0.00,986.50,49,0,12.20,24.13,802,300,14.55,9.90
14,13,4,JFK,14,4,33,50,82,7.00,0.50,...,0.62,992.50,67,0,15.70,24.13,301,213,15.85,10.79
16,11,4,MCO,14,4,35,20,82,7.00,0.50,...,0.00,"1,018.00",60,0,21.30,24.13,801,305,8.75,5.63
18,18,5,ORD,21,5,52,47,75,1.10,0.00,...,0.00,981.70,47,0,16.60,24.13,802,206,8.91,5.81


In [23]:
pred_data2.to_csv('pred_data2.csv')

In [24]:
len(pred_data2)

12